In [29]:
import pandas as pd
import numpy as np
from dateutil import parser
import psycopg

In [30]:
df=pd.read_csv("~/health_pipline/healthcare_dataset.csv")

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  str    
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  str    
 3   Blood Type          55500 non-null  str    
 4   Medical Condition   55500 non-null  str    
 5   Date of Admission   55500 non-null  str    
 6   Doctor              55500 non-null  str    
 7   Hospital            55500 non-null  str    
 8   Insurance Provider  55500 non-null  str    
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  str    
 12  Discharge Date      55500 non-null  str    
 13  Medication          55500 non-null  str    
 14  Test Results        55500 non-null  str    
dtypes: float64(1), int64(2), str(12)
memory usage: 6.4 MB


In [32]:
df.dtypes


Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition         str
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object

In [33]:
df.isnull().sum()


Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

In [34]:
df.duplicated().sum()

np.int64(534)

In [35]:
df.describe()


,Age,Billing Amount,Room Number
count,55500.000000,55500.000000,55500.000000
mean,51.539459,25539.316097,301.134829
std,19.602454,14211.454431,115.243069
min,13.000000,-2008.492140,101.000000
25%,35.000000,13241.224652,202.000000
50%,52.000000,25538.069376,302.000000
75%,68.000000,37820.508436,401.000000
max,89.000000,52764.276736,500.000000


In [36]:
df["Name"].sample(20)
df["Gender"].unique()
df["Blood Type"].unique()
df["Medical Condition"].unique()
df["Doctor"].sample(20)
df["Hospital"].sample(20)
df["Insurance Provider"].unique()
df["Admission Type"].unique()
df["Medication"].unique()
df["Test Results"].unique()

<StringArray>
['Normal', 'Inconclusive', 'Abnormal']
Length: 3, dtype: str

In [37]:
#remove the duplicates 
df=df.drop_duplicates()
df.duplicated().sum()


np.int64(0)

In [38]:
#Change the categorical columns to lowercase
df_col=["Name","Gender","Blood Type","Medical Condition","Doctor","Hospital","Insurance Provider",\
    "Admission Type","Medication","Test Results"]

for col in df_col:
    df[col]=df[col].str.lower().str.strip()

In [39]:
#Handling the negaitve billing amounts
df["Billing Amount"]=df["Billing Amount"].abs()

In [40]:
#Changing the Date columns to date types
df["Date of Admission"]=pd.to_datetime(df["Date of Admission"])
df["Discharge Date"]=pd.to_datetime(df["Discharge Date"])

In [41]:
df.dtypes

Name                             str
Age                            int64
Gender                           str
Blood Type                       str
Medical Condition                str
Date of Admission     datetime64[us]
Doctor                           str
Hospital                         str
Insurance Provider               str
Billing Amount               float64
Room Number                    int64
Admission Type                   str
Discharge Date        datetime64[us]
Medication                       str
Test Results                     str
dtype: object

In [42]:
#Handling patient name (firstName,lastName)
df[["patient_title","name"]]=df["Name"].str.extract(r'^(?:([A-Za-z]+)\.\s*)?(.*)')
df[["first","middle","last"]]=df["Name"].str.extract(r'([A-Za-z]+)\s+([A-Za-z]\.)?\s*([A-Za-z]+)')
df["Name"]=df["first"] + " " + df["last"]
# drop the reduntant columns 
df=df.drop(["name","first","middle","last"],axis=1)


In [43]:
#Handling Doctor name (firstName,lastName)
df[["doctor_title","name"]]=df["Doctor"].str.extract(r'^(?:([A-Za-z]+)\.\s*)?(.*)')
df[["first","middle","last"]]=df["Doctor"].str.extract(r'([A-Za-z]+)\s+([A-Za-z]\.)?\s*([A-Za-z]+)')
df["Doctor"]=df["first"] + " " + df["last"]
# drop the reduntant columns 
df=df.drop(["name","first","middle","last","doctor_title"],axis=1)

In [44]:
#handling Hospital names
df["Hospital"] = df["Hospital"].str.replace(r"\band\b|,", "", regex=True)
df["Hospital"] = df["Hospital"].str.replace(r"\s+|-", " ", regex=True)
df["Hospital"] = df["Hospital"].str.strip()

In [45]:
#handling column names
df.columns=df.columns.str.lower().str.replace(' ','_')
df

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,patient_title
0,bobby jackson,30,male,b-,cancer,2024-01-31,matthew smith,sons miller,blue cross,18856.281306,328,urgent,2024-02-02,paracetamol,normal,NaN
1,leslie terry,62,male,a+,obesity,2019-08-20,samantha davies,kim inc,medicare,33643.327287,265,emergency,2019-08-26,ibuprofen,inconclusive,NaN
2,danny smith,76,female,a-,obesity,2022-09-22,tiffany mitchell,cook plc,aetna,27955.096079,205,emergency,2022-10-07,aspirin,normal,NaN
3,andrew watts,28,female,o+,diabetes,2020-11-18,kevin wells,hernandez rogers vang,medicare,37909.782410,450,elective,2020-12-18,ibuprofen,abnormal,NaN
4,adrienne bell,43,female,ab+,cancer,2022-09-19,kathleen hanna,white white,aetna,14238.317814,458,urgent,2022-10-09,penicillin,abnormal,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,elizabeth jackson,42,female,o+,asthma,2020-08-16,joshua jarvis,jones thompson,blue cross,2650.714952,417,elective,2020-09-15,penicillin,abnormal,NaN
55496,kyle perez,61,female,ab-,obesity,2020-01-23,taylor sullivan,tucker moyer,cigna,31457.797307,316,elective,2020-02-01,aspirin,normal,NaN
55497,heather wang,38,female,b+,hypertension,2020-07-13,joe jacobs,mahoney johnson vasquez,unitedhealthcare,27620.764717,347,urgent,2020-08-10,ibuprofen,abnormal,NaN
55498,jennifer jones,43,male,o-,arthritis,2019-05-25,kimberly curry,jackson todd castro,medicare,32451.092358,321,elective,2019-05-31,ibuprofen,abnormal,NaN


In [46]:
df.to_csv("~/health_pipline/healthcare_cleaned.csv",index=False)

In [47]:
conn_str = "dbname=medical user=postgres password=postgres host=localhost port=5432"
doctor_df = df[["doctor"]].drop_duplicates()
hospital_df = df[["hospital"]].drop_duplicates()
patient_df = df[["name","age", "gender", "blood_type", "insurance_provider","patient_title"]].drop_duplicates(subset=["name","age","gender","blood_type","insurance_provider"])

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:

        # doctor
        with cur.copy("COPY doctor (name) FROM STDIN") as copy:
            for row in doctor_df.itertuples(index=False, name=None):
                copy.write_row(row)

        # hospital
        with cur.copy("COPY hospital (name) FROM STDIN") as copy:
            for row in hospital_df.itertuples(index=False, name=None):
                copy.write_row(row)

        # patient
        with cur.copy("COPY patient (name,age, gender, blood_type, insurance_provider,patient_title) FROM STDIN") as copy:
            for row in patient_df.itertuples(index=False, name=None):
                copy.write_row(row)
        

In [48]:
with psycopg.connect(conn_str) as conn:
    patient=pd.read_sql_query("""select * from patient""",conn)
    doctor=pd.read_sql_query("""select * from doctor""",conn)
    hospital=pd.read_sql_query("""select * from hospital""",conn)
   

/tmp/ipykernel_12704/1860832911.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  patient=pd.read_sql_query("""select * from patient""",conn)
/tmp/ipykernel_12704/1860832911.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  doctor=pd.read_sql_query("""select * from doctor""",conn)
/tmp/ipykernel_12704/1860832911.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  hospital=pd.read_sql_query("""select * from hospital""",conn)


In [49]:
doctor = doctor.rename(
    columns={
        "doctor_id": "doctor_id",
        "name": "doctor_name"
    }
)

hospital = hospital.rename(
    columns={
        "hospital_id": "hospital_id",
        "name": "hospital_name"
    }
)

patient = patient.rename(
    columns={
        "patient_id": "patient_id",
        "name": "patient_name"
    }
)


In [50]:
df = df.merge(
    doctor,
    left_on="doctor",
    right_on="doctor_name",
    how="left"
)

In [51]:
df = df.merge(
    hospital,
    left_on="hospital",
    right_on="hospital_name",
    how="left"
)

In [52]:
df

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,patient_title,doctor_id,doctor_name,hospital_id,hospital_name
0,bobby jackson,30,male,b-,cancer,2024-01-31,matthew smith,sons miller,blue cross,18856.281306,328,urgent,2024-02-02,paracetamol,normal,NaN,1,matthew smith,1,sons miller
1,leslie terry,62,male,a+,obesity,2019-08-20,samantha davies,kim inc,medicare,33643.327287,265,emergency,2019-08-26,ibuprofen,inconclusive,NaN,2,samantha davies,2,kim inc
2,danny smith,76,female,a-,obesity,2022-09-22,tiffany mitchell,cook plc,aetna,27955.096079,205,emergency,2022-10-07,aspirin,normal,NaN,3,tiffany mitchell,3,cook plc
3,andrew watts,28,female,o+,diabetes,2020-11-18,kevin wells,hernandez rogers vang,medicare,37909.782410,450,elective,2020-12-18,ibuprofen,abnormal,NaN,4,kevin wells,4,hernandez rogers vang
4,adrienne bell,43,female,ab+,cancer,2022-09-19,kathleen hanna,white white,aetna,14238.317814,458,urgent,2022-10-09,penicillin,abnormal,NaN,5,kathleen hanna,5,white white
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54961,elizabeth jackson,42,female,o+,asthma,2020-08-16,joshua jarvis,jones thompson,blue cross,2650.714952,417,elective,2020-09-15,penicillin,abnormal,NaN,22330,joshua jarvis,21662,jones thompson
54962,kyle perez,61,female,ab-,obesity,2020-01-23,taylor sullivan,tucker moyer,cigna,31457.797307,316,elective,2020-02-01,aspirin,normal,NaN,801,taylor sullivan,793,tucker moyer
54963,heather wang,38,female,b+,hypertension,2020-07-13,joe jacobs,mahoney johnson vasquez,unitedhealthcare,27620.764717,347,urgent,2020-08-10,ibuprofen,abnormal,NaN,1865,joe jacobs,1839,mahoney johnson vasquez
54964,jennifer jones,43,male,o-,arthritis,2019-05-25,kimberly curry,jackson todd castro,medicare,32451.092358,321,elective,2019-05-31,ibuprofen,abnormal,NaN,22556,kimberly curry,21891,jackson todd castro


In [53]:
df = df.merge(
    patient,
    left_on=[
        "name",
        "age",
        "blood_type",
        "insurance_provider"
    ],
    right_on=[
        "patient_name",
        "age",
        "blood_type",
        "insurance_provider"
    ],
    how="left"
)

In [54]:
df

,name,age,gender_x,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,...,test_results,patient_title_x,doctor_id,doctor_name,hospital_id,hospital_name,patient_id,patient_name,gender_y,patient_title_y
0,bobby jackson,30,male,b-,cancer,2024-01-31,matthew smith,sons miller,blue cross,18856.281306,...,normal,NaN,1,matthew smith,1,sons miller,1,bobby jackson,male,nan
1,leslie terry,62,male,a+,obesity,2019-08-20,samantha davies,kim inc,medicare,33643.327287,...,inconclusive,NaN,2,samantha davies,2,kim inc,2,leslie terry,male,nan
2,danny smith,76,female,a-,obesity,2022-09-22,tiffany mitchell,cook plc,aetna,27955.096079,...,normal,NaN,3,tiffany mitchell,3,cook plc,3,danny smith,female,nan
3,andrew watts,28,female,o+,diabetes,2020-11-18,kevin wells,hernandez rogers vang,medicare,37909.782410,...,abnormal,NaN,4,kevin wells,4,hernandez rogers vang,4,andrew watts,female,nan
4,adrienne bell,43,female,ab+,cancer,2022-09-19,kathleen hanna,white white,aetna,14238.317814,...,abnormal,NaN,5,kathleen hanna,5,white white,5,adrienne bell,female,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54971,elizabeth jackson,42,female,o+,asthma,2020-08-16,joshua jarvis,jones thompson,blue cross,2650.714952,...,abnormal,NaN,22330,joshua jarvis,21662,jones thompson,54959,elizabeth jackson,female,nan
54972,kyle perez,61,female,ab-,obesity,2020-01-23,taylor sullivan,tucker moyer,cigna,31457.797307,...,normal,NaN,801,taylor sullivan,793,tucker moyer,54960,kyle perez,female,nan
54973,heather wang,38,female,b+,hypertension,2020-07-13,joe jacobs,mahoney johnson vasquez,unitedhealthcare,27620.764717,...,abnormal,NaN,1865,joe jacobs,1839,mahoney johnson vasquez,54961,heather wang,female,nan
54974,jennifer jones,43,male,o-,arthritis,2019-05-25,kimberly curry,jackson todd castro,medicare,32451.092358,...,abnormal,NaN,22556,kimberly curry,21891,jackson todd castro,54962,jennifer jones,male,nan


In [55]:
df_visit=df[["patient_id","doctor_id","hospital_id","date_of_admission","discharge_date","admission_type","medical_condition","billing_amount","medication","test_results","room_number"]]

In [56]:
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        with cur.copy("COPY visit (patient_id,doctor_id,hospital_id,date_of_admission,discharge_date,admission_type,medical_condition,billing_amount,medication,test_results,room_number) FROM STDIN") as copy:
            for row in df_visit.itertuples(index=False, name=None):
                copy.write_row(row)

In [57]:
df.sample(50)

,name,age,gender_x,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,...,test_results,patient_title_x,doctor_id,doctor_name,hospital_id,hospital_name,patient_id,patient_name,gender_y,patient_title_y
51549,michael sandoval,72,female,ab-,cancer,2023-07-27,scott hayes,sons bond,aetna,40932.204414,...,inconclusive,NaN,6098,scott hayes,5594,sons bond,51538,michael sandoval,female,nan
53222,samuel maldonado,42,female,o+,obesity,2023-10-09,tara kennedy,lopez lewis carr,aetna,43316.385739,...,abnormal,NaN,38260,tara kennedy,37762,lopez lewis carr,53211,samuel maldonado,female,nan
6806,candace schroeder,23,female,a-,asthma,2020-09-23,eric ball,anderson norris,cigna,47854.586015,...,abnormal,NaN,6497,eric ball,6311,anderson norris,6806,candace schroeder,female,nan
45428,stephen morrow,79,female,ab+,asthma,2020-07-22,rebecca morales,brown garcia,cigna,6467.062334,...,normal,NaN,36586,rebecca morales,35980,brown garcia,45419,stephen morrow,female,nan
30020,dustin alexander,69,male,b-,hypertension,2022-10-18,paul solis,bradshaw herrera cook,blue cross,16732.225643,...,inconclusive,NaN,25610,paul solis,24875,bradshaw herrera cook,30016,dustin alexander,male,nan
39398,sarah torres,67,male,ab-,cancer,2019-07-03,calvin nunez,roberts diaz,medicare,39117.954611,...,abnormal,NaN,32442,calvin nunez,31544,roberts diaz,39391,sarah torres,male,nan
49711,grace garcia,29,male,ab+,asthma,2020-02-27,kendra graves,myers west,aetna,14385.167817,...,abnormal,NaN,39498,kendra graves,39060,myers west,49701,grace garcia,male,nan
29131,allison sawyer,63,female,b-,asthma,2022-03-11,joshua wilson,bell ortiz,aetna,3275.584877,...,normal,NaN,4148,joshua wilson,24236,bell ortiz,29127,allison sawyer,female,nan
48995,michael henry,56,male,b+,arthritis,2023-01-06,wesley barnes,lloyd parker barrett,blue cross,13754.577943,...,inconclusive,NaN,39013,wesley barnes,38533,lloyd parker barrett,48985,michael henry,male,nan
8700,debbie thomas,28,female,o-,diabetes,2020-08-08,nicholas mack,nguyen monroe gomez,aetna,34105.143572,...,abnormal,ms,8214,nicholas mack,7941,nguyen monroe gomez,8700,debbie thomas,female,ms
